In [14]:
import os
import pandas as pd

def get_folders_after_threshold(base_path, csv_path, skip_list, threshold=1000):
    # 1. Load the CSV and create a mapping of address -> order (index)
    try:
        df = pd.read_csv(csv_path)
        # We create a dictionary for O(1) lookup speed
        # Assumes 'address' column contains the folder names
        order_map = {val: i for val, i in zip(df['address'], df["transaction_count"])}
    except Exception as e:
        return f"Error reading CSV: {e}"

    # 2. Get subfolders and filter out the skip_list
    try:
        all_entries = os.listdir(base_path)
        subfolders = [
            f for f in all_entries 
            if os.path.isdir(os.path.join(base_path, f)) and f not in skip_list
        ]
    except Exception as e:
        return f"Error accessing directory: {e}"

    # 3. Pair folders with their rank from the CSV
    # Folders not in the CSV are ignored or could be handled separately
    ranked_folders = []
    for folder in subfolders:
        if folder in order_map:
            ranked_folders.append((folder, order_map[folder]))
    
    # 4. Sort based on the CSV index (the second element in the tuple)
    ranked_folders.sort(key=lambda x: -x[1])

    # 5. Extract folders appearing after the 1000th position
    # Python is 0-indexed, so the 1001st item is at index 1000
    after_threshold = [folder for folder, rank in ranked_folders[threshold:]]

    return after_threshold

# Example Usage:
# result = get_folders_after_threshold("./my_data", "registry.csv", ["temp", "logs"])
# print(f"Folders after the 1000th: {result}")

In [15]:
to_erase = get_folders_after_threshold("compiled", "contract_list_all.csv", ["0x8355dbe8b0e275abad27eb843f3eaf3fc855e525",
"0x8ee5dd62a654a60f6f17a99d544102f37b58da26",
"0xa80f2c8f61c56546001f5fc2eb8d6e4e72c45d4c",
"0x0c2e57efddba8c768147d1fdf9176a0a6ebd5d83",
"0x3caca7b48d0573d793d3b0279b5f0029180e83b6",
"0x15b543e986b8c34074dfc9901136d9355a537e7e",
])

In [18]:
import shutil
from pathlib import Path

for folder in to_erase:
    shutil.rmtree(Path("compiled").joinpath(folder))